In [1]:
import boto3
import json
import numpy as np
from pathlib import Path
import sys
import tempfile

sys.path.append("../src")

from embeddings import load_embedding_model
from retrieval import retrieve_top_k_chunks

#### S3 paths

In [2]:
BUCKET_NAME = "multiomic-vae-literature-rag-123223178042-eu-north-1-an"

EMBEDDINGS_KEY = "embeddings/chunk_embeddings.npy"
METADATA_KEY = "embeddings/chunk_metadata.jsonl"

s3 = boto3.client("s3")

#### Load embeddings and metadata from S3

In [3]:
with tempfile.TemporaryDirectory() as tmpdir:
    embeddings_path = Path(tmpdir) / "chunk_embeddings.npy"
    metadata_path = Path(tmpdir) / "chunk_metadata.jsonl"

    s3.download_file(
        BUCKET_NAME,
        EMBEDDINGS_KEY,
        str(embeddings_path)
    )

    s3.download_file(
        BUCKET_NAME,
        METADATA_KEY,
        str(metadata_path)
    )

    chunk_embeddings = np.load(embeddings_path)

    chunk_metadata = [
        json.loads(line)
        for line in metadata_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

print("Embeddings shape:", chunk_embeddings.shape)
print("Metadata count:", len(chunk_metadata))

Embeddings shape: (660, 384)
Metadata count: 660


#### Load embedding model

In [5]:
model = load_embedding_model(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

#### Retrieve relevant chunks

In [6]:
question = "Which papers use variational autoencoders for multi-omics integration?"

results = retrieve_top_k_chunks(
    question=question,
    model=model,
    chunk_embeddings=chunk_embeddings,
    chunk_metadata=chunk_metadata,
    top_k=5,
)

#### Print retrieved chunks

In [7]:
for result in results:
    print("=" * 100)
    print("Rank:", result["rank"])
    print("Score:", result["score"])
    print("Paper:", result["paper_name"])
    print("Chunk ID:", result["chunk_id"])
    print()
    print(result["text"][:1200])
    print()

Rank: 1
Score: 0.5976897478103638
Paper: factVAE
Chunk ID: 0

Received: January 15, 2025. Revised: March 02, 2025. Accepted: March 21, 2025 © The Author(s) 2025. Published by Oxford University Press. This is an Open Access article distributed under the terms of the Creative Commons Attribution Non-Commercial License ( https://creativecommons.org/ licenses/by-nc/4.0/), which permits non-commercial re-use, distribution, and reproduction in any medium, provided the original work is properly cited. For commercial re-use, please contact journals.permissions@oup.com Briefings in Bioinformatics , 2025, 26(2), bbaf157 https://doi.org/10.1093/bib/bbaf157 Problem Solving Protocol FactVAE: a factorized variational autoencoder for single-cell multi-omics data integration analysis Linjie Wang 1, Huixia Zhang1, Bo Yi1, Weidong Xie 1, Kun Yu2, Wei Li3,4,*, Keqin Li5, Dazhe Zhao1,* 1 School of Computer Science and Engineering, Northeastern University , 110819, Shenyang, China 2 College of Medicine and